In [1]:
!pip install -U bitsandbytes

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 59.4/59.4 MB 11.9 MB/s eta 0:00:00


In [2]:
import os, re, random, math
import pandas as pd
import torch

from transformers import (
    AutoTokenizer, AutoModelForCausalLM,
    BitsAndBytesConfig, LogitsProcessor, LogitsProcessorList
)

import bitsandbytes as bnb

In [3]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [4]:
Model = "mistralai/Mistral-7B-Instruct-v0.3"
OUTPUT_DIR = "./mistral_cls_adapter"
os.makedirs(OUTPUT_DIR, exist_ok=True)

Train_path = "/content/drive/MyDrive/Colab Notebooks/DataSci_266_NLP/Final_Project/Mental_Health/Train_Test_Val_data/train_balanced.csv"
Val_path = "/content/drive/MyDrive/Colab Notebooks/DataSci_266_NLP/Final_Project/Mental_Health/Train_Test_Val_data/val.csv"
# Test_path = "/content/drive/MyDrive/Colab Notebooks/DataSci_266_NLP/Final_Project/Mental_Health/Train_Test_Val_data/test.csv"

K_PER_LABEL = 3
MAX_VAL_SAMPLES = 200
USE_4BIT = True

Labels = ["Normal", "Bipolar/Personality", "Anxiety/Stress", "Depressive_Spectrum"]


In [5]:
train_df = pd.read_csv(Train_path)[["statement", "status_combined"]].dropna()
val_df   = pd.read_csv(Val_path)[["statement", "status_combined"]].dropna()

In [6]:
shots = []
for lab in Labels:
    subset = train_df[train_df["status_combined"] == lab]
    n = min(K_PER_LABEL, len(subset))
    shots.extend(subset.sample(n=n, random_state=42).to_dict("records"))
random.shuffle(shots)

print(f"[Data] Using {len(shots)} few-shot examples total ({K_PER_LABEL} per label requested).")

[Data] Using 12 few-shot examples total (3 per label requested).


In [7]:
bnb = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_use_double_quant=True,
    bnb_4bit_compute_dtype=torch.float16
)

tokenizer = AutoTokenizer.from_pretrained(Model)

model = AutoModelForCausalLM.from_pretrained(
        Model, device_map="auto", quantization_config=bnb
    )

model.eval()

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.model:   0%|          | 0.00/587k [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/414 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/601 [00:00<?, ?B/s]

...

model-00001-of-00003.safetensors:   0%|          | 0.00/4.95G [00:00<?, ?B/s]

model-00002-of-00003.safetensors:   0%|          | 0.00/5.00G [00:00<?, ?B/s]

Loading checkpoint shards:   0%|          | 0/3 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

MistralForCausalLM(
  (model): MistralModel(
    (embed_tokens): Embedding(32768, 4096)
    (layers): ModuleList(
      (0-31): 32 x MistralDecoderLayer(
        (self_attn): MistralAttention(
          (q_proj): Linear4bit(in_features=4096, out_features=4096, bias=False)
          (k_proj): Linear4bit(in_features=4096, out_features=1024, bias=False)
          (v_proj): Linear4bit(in_features=4096, out_features=1024, bias=False)
          (o_proj): Linear4bit(in_features=4096, out_features=4096, bias=False)
        )
        (mlp): MistralMLP(
          (gate_proj): Linear4bit(in_features=4096, out_features=14336, bias=False)
          (up_proj): Linear4bit(in_features=4096, out_features=14336, bias=False)
          (down_proj): Linear4bit(in_features=14336, out_features=4096, bias=False)
          (act_fn): SiLUActivation()
        )
        (input_layernorm): MistralRMSNorm((4096,), eps=1e-05)
        (post_attention_layernorm): MistralRMSNorm((4096,), eps=1e-05)
      )
    )
    (n

In [8]:
SYSTEM_PROMPT = "You are a an expert classifier. Reply with exactly one label from: " + ", ".join(Labels) + "."

def build_fewshot_block(examples):
    lines = []
    for i, ex in enumerate(examples, 1):
        s = ex["statement"].strip().replace("\n", " ")
        y = ex["status_combined"]
        lines.append(f"Example {i}:\nText: {s}\nLabel options: " + ", ".join(Labels) + f"\nAnswer: {y}\n")
    return "\n".join(lines)

FEWSHOT_TEXT = build_fewshot_block(shots)

def make_messages(text):
    user = (
        FEWSHOT_TEXT
        + "\n---\n"
        + f"Now classify the following.\nText: {text}\nLabel options: " + ", ".join(Labels) + "\nAnswer:"
    )
    return [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": user},
    ]

In [9]:
class OnlyLabelTokens(LogitsProcessor):
    def __init__(self, allowed_ids):
        self.allowed = torch.tensor(allowed_ids, dtype=torch.long)
    def __call__(self, input_ids, scores):
        mask = torch.full_like(scores, float("-inf"))
        mask[:, self.allowed] = 0.0
        return scores + mask

def first_token_ids_for(labels):
    ids = set()
    for lab in labels:
        toks = tokenizer(lab, add_special_tokens=False).input_ids
        if toks:
            ids.add(toks[0])
    return sorted(list(ids))

allowed = first_token_ids_for(Labels)
logits_processor = LogitsProcessorList([OnlyLabelTokens(allowed)])

In [18]:
def predict_label(text):
    messages = make_messages(text)
    inputs = tokenizer.apply_chat_template(
        messages,
        add_generation_prompt=True,
        return_tensors="pt"
        )

    if isinstance(inputs, torch.Tensor):  # some tokenizers return only input_ids
        input_ids = inputs.to(model.device)
        attention_mask = torch.ones_like(input_ids)  # avoid attention_mask warnings
        input_kwargs = {"input_ids": input_ids, "attention_mask": attention_mask}
        prompt_len = input_ids.shape[-1]
    else:
        input_kwargs = {k: v.to(model.device) for k, v in inputs.items()}
        # ensure attention_mask exists
        if "attention_mask" not in input_kwargs:
            input_kwargs["attention_mask"] = torch.ones_like(input_kwargs["input_ids"])
        prompt_len = input_kwargs["input_ids"].shape[-1]

    with torch.no_grad():
        out = model.generate(
            **input_kwargs,
            max_new_tokens=3,
            do_sample=False,
            num_beams=1,
            logits_processor=logits_processor,
            eos_token_id=tokenizer.eos_token_id,
        )
    gen = tokenizer.decode(out[0][inputs.shape[-1]:], skip_special_tokens=True).strip()
    # Normalize to the first matching label
    for lab in Labels:
        if gen.startswith(lab):
            return lab
    # fallback: fuzzy contains
    for lab in Labels:
        if lab.lower() in gen.lower():
            return lab
    return gen

In [14]:
# Load data
df = pd.read_csv(VAL_PATH if SPLIT == "val" else TEST_PATH)[["statement", "status_combined"]].dropna()
if MAX_SAMPLES > 0:
    df = df.sample(n=min(MAX_SAMPLES, len(df)), random_state=123).reset_index(drop=True)

# Predict
preds = []
for i, row in df.iterrows():
    preds.append(predict_label(row["statement"]))  # uses your few-shot function
    if (i + 1) % 20 == 0:
        print(f"[{SPLIT}] processed {i + 1}/{len(df)}")

y_true = df["status_combined"].tolist()
y_pred = preds

# Accuracy + per-class report
acc = accuracy_score(y_true, y_pred)
print(f"\n{SPLIT.upper()} accuracy: {acc:.4f}\n")
print("Classification report:")
print(classification_report(y_true, y_pred, labels=LABELS, digits=4))

# Confusion matrix
cm = confusion_matrix(y_true, y_pred, labels=LABELS)

# Plot (default matplotlib styles)
fig = plt.figure(figsize=(6, 5))
plt.imshow(cm, interpolation="nearest")
plt.title(f"Confusion Matrix ({SPLIT})")
plt.xlabel("Predicted")
plt.ylabel("True")
plt.xticks(np.arange(len(LABELS)), LABELS, rotation=45, ha="right")
plt.yticks(np.arange(len(LABELS)), LABELS)

# Annotate counts
for i in range(cm.shape[0]):
    for j in range(cm.shape[1]):
        plt.text(j, i, str(cm[i, j]),
                 ha="center", va="center")

plt.tight_layout()
plt.show()

# Save predictions for audit
out_csv = f"/mnt/data/fewshot_predictions_{SPLIT}.csv"
pd.DataFrame({"statement": df["statement"], "gold": y_true, "pred": y_pred}).to_csv(out_csv, index=False)
print(f"Saved predictions → {out_csv}")


NameError: name 'SPLIT' is not defined

In [17]:
if MAX_VAL_SAMPLES > 0:
    val_sample = val_df.sample(n=min(MAX_VAL_SAMPLES, len(val_df)), random_state=123).reset_index(drop=True)
else:
    val_sample = val_df.reset_index(drop=True)

preds = []
for i, row in val_sample.iterrows():
    yhat = predict_label(row["statement"])
    preds.append(yhat)
    if (i+1) % 20 == 0:
        print(f"[Eval] Processed {i+1}/{len(val_sample)}")

gold = val_sample["status_combined"].tolist()
acc = sum(p==g for p,g in zip(preds, gold)) / len(gold)
print(f"\nFew-shot baseline accuracy: {acc:.4f} on {len(gold)} samples")

# Confusion-like summary
from collections import Counter
counts = Counter(zip(gold, preds))
print("\nTop (gold -> pred) pairs:")
for (g,p), n in counts.most_common(20):
    print(f"{g:22s} -> {p:22s} : {n}")


The attention mask and the pad token id were not set. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.


RuntimeError: Expected all tensors to be on the same device, but got index is on cpu, different from other tensors on cuda:0 (when checking argument in method wrapper_CUDA__index_select)